# Derivative Pricing with Black-Scholes inputs - implied vol surface

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import datetime
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from DerivModel import train_model, FeedForwardNetwork, black_scholes
from DerivMetrics import evaluate_metrics, compute_metrics 
from DerivPlots import scatter_plot, plot_errors
import json
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from tqdm.notebook import tqdm, trange

## 20-day Historical Vol

### Load and Prepare the dataframe

In [ ]:
in_file = 'SPXOptions.csv'
dfopt = pd.read_csv(in_file)

In [ ]:
dfopt

In [ ]:
max_strike = dfopt['strike'].max()
print(max_strike)

In [ ]:
min_strike = dfopt['strike'].min()
print(min_strike)

In [ ]:
min_hist_vol = dfopt['20D_HV_x'].min()
print(min_hist_vol)

In [ ]:
max_hist_vol = dfopt['20D_HV_x'].max()
print(max_hist_vol)

In [ ]:
min_rate = dfopt['rate'].min()
print(min_rate)

In [ ]:
max_rate = dfopt['rate'].max()
print(max_rate)

In [ ]:
min_spot = dfopt['underlying'].min()
print(min_spot)

In [ ]:
max_spot = dfopt['underlying'].max()
print(max_spot)

In [ ]:
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

In [ ]:
#Separate into X and y
X = dfopt[['type', 'strike', '20D_HV_x', 'maturity', 'rate', 'underlying']]
y = dfopt['mid']

In [ ]:
X['type'] = X['type'].replace({'call': 1, 'put': 0})

In [ ]:
# Split data into train and a temporary set (tmp) first
X_train, us_test, y_train, y_test = train_test_split(X, y, test_size=0.2) 

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(us_test)

In [ ]:
torch_tensor = torch.from_numpy(X_train).float().to(device)
y_tensor = torch.from_numpy(y_train.to_numpy().copy()).float().to(device)
dataset = TensorDataset(torch_tensor, y_tensor)
batch_size = 512
data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

test_torch_tensor = torch.from_numpy(X_test).float().to(device)
test_y_tensor = torch.from_numpy(y_test.to_numpy().copy()).float().to(device)
test_dataset = TensorDataset(test_torch_tensor, test_y_tensor)
test_data_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=True)

### Build and Train model

In [ ]:
epochs = 50
no_layers = 5
no_neurons = 4096
lr = 0.001

In [ ]:
model = FeedForwardNetwork(input_size=6, 
                           num_hidden_layers=no_layers, 
                           neurons_per_layer=no_neurons).to(device)
optimizer = optim.Adam(model.parameters(), lr=lr)
loss_fn = torch.nn.MSELoss()

In [ ]:
train_errors, test_errors = train_model(model, data_loader, test_data_loader, 
                                        loss_fn, optimizer, epochs)

In [ ]:
metrics, all_results, all_targets = evaluate_metrics(test_data_loader, model)

In [ ]:
metrics

## Generate the Vol Surface

In [ ]:
hist_vol = 0.10
spot = 1500
x = np.arange(1000.0, 2100.0, 100.0)
y = np.arange(0.25, 3.0, 0.25)

In [ ]:
rate = 0.0010

In [ ]:
vols = np.zeros((len(x), len(y)))

In [ ]:
import math
from scipy.stats import norm

def implied_volatility(S, K, T, r, C_market, option_type, tol=1e-8, max_iterations=500):
    """
    Calculate the implied volatility using the Newton-Raphson method.
    """
    sigma = 0.2  # Initial guess
    for i in range(max_iterations):
        #print(option_type, K, sigma, T, spot, r)
        C_model = black_scholes(option_type, K, sigma, T, spot, r)
        
        diff = C_model.value() - C_market
        
        if abs(diff) < tol:
            return sigma
        
        if C_model.vega(T) == 0.0:  # To avoid division by zero
            return sigma
        
        sigma = sigma - diff/C_model.vega(T)

        # Ensure sigma remains positive
        sigma = max(sigma, 1e-5)

    return sigma  # Return latest guess if not converged

In [ ]:
for ix, xv in enumerate(x):
    for iy, yv in enumerate(y):
        optype = 1 # call
        if(ix < spot): 
            optype = 0
        inarr = [[optype, xv, hist_vol, yv, rate, spot]]
        cols = ['type', 'strike', '20D_HV_x', 'maturity', 'rate', 'underlying']
        df = pd.DataFrame(inarr, columns=cols) 
        scdata = scaler.transform(df)
        tinlst = torch.tensor(scdata).float().to(device)
        bsprice = model(tinlst).item()
        vols[ix, iy] = implied_volatility(spot, xv, yv, rate, bsprice, optype)

In [ ]:
vols

In [ ]:
def plot_3d_surface(x, y, z, filename, rotation_angle=0):
    # Create a meshgrid for the x and y values
    X, Y = np.meshgrid(x, y)

    # Create the figure and 3D axis
    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(111, projection='3d')

    # Plot the surface
    ax.plot_surface(X, Y, z, cmap='viridis')

    # Set labels
    ax.set_xlabel('Strike')
    ax.set_ylabel('Maturity')
    ax.set_zlabel('Volatility')
    
    # Rotate the plot by the specified angle
    ax.view_init(azim=rotation_angle)

    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    
    # Show the plot
    plt.show()

In [ ]:
filename = 'VolSurface.png'
rotation_angle = 45
plot_3d_surface(x, y, vols, filename, rotation_angle)